# 02 - Motif Enrichment And Chain Signatures

## Purpose

This notebook asks whether motif and chain organization is stronger than expected from broad graph structure alone. It joins two complementary views:

- weighted triad class-share deviations against a degree/dyad-preserving null model;
- directed simple 3-node E/I chain signatures (`X -> Y -> Z`) with regional and cell-type pathway breakdowns.

The previous scalar weighted-triad-participation measure is retained only as a diagnostic because it is mathematically degenerate: it reduces to node strength plus a constant.

## Outputs

- `outputs/consolidated_02_enrichment_chains/weighted_null/`
- `outputs/consolidated_02_enrichment_chains/chain_signatures/`

In [ ]:
from pathlib import Path
from IPython.display import display

from ei_chain_signature import ChainSignatureConfig, run_chain_signature_analysis
from motif_workflow import ConnectomePaths
from weighted_motif_null import run_weighted_motif_null_analysis

paths = ConnectomePaths(matrix=Path("mij_matrix.csv"), netlist=Path("mij_netlist.csv"))
output_root = Path("outputs") / "consolidated_02_enrichment_chains"

N_NULLS = 100
RANDOM_SEED = 11

## Weighted Triad Class Shares Against Nulls

In [ ]:
weighted = run_weighted_motif_null_analysis(
    nulls=N_NULLS,
    seed=RANDOM_SEED,
    matrix=paths.matrix,
    netlist=paths.netlist,
    outdir=output_root / "weighted_null",
    progress=True,
)

print(f"Weighted-null outputs: {weighted['output_dir'].resolve()}")

In [ ]:
display(weighted["chain_signature_table"])
display(
    weighted["strong_class_share_deviations"]
    .sort_values("z_score", key=lambda s: s.abs(), ascending=False)
    .head(25)
)

## E/I Chain Signatures

The chain-signature analysis keeps exact high-weight pathways for audit while also summarizing region, cell-type, and role-specific composition.

In [ ]:
chain_config = ChainSignatureConfig(
    data_path=paths.netlist,
    output_dir=output_root / "chain_signatures",
    selected_signature="EII",
    top_n=20,
)

chains = run_chain_signature_analysis(chain_config, make_figures=True, show_figures=True)
print(f"Chain-signature tables: {chain_config.table_dir.resolve()}")
print(f"Chain-signature figures: {chain_config.figure_dir.resolve()}")

In [ ]:
display(chains.tables["data_profile"])
display(chains.tables["chain_profile"])
display(chains.tables["signature_summary"])

In [ ]:
display(chains.tables["region_pathways_by_signature"].head(20))
display(chains.tables["cell_type_pathways_by_signature"].head(20))
display(chains.tables[f"high_weight_{chain_config.selected_signature}_chains"].head(20))

## Handoff To Notebook 03

This notebook identifies motif classes, E/I signatures, and cell/region pathways worth treating as candidates. Notebook 03 asks the dynamical question: do the same motif families carry local Schur modes, signed motif neighborhoods, or stability effects?